            # Week 4 - Data Preparation for Smart Driver Monitoring

            This notebook prepares the driving-behavior motion dataset for Week 4 of the internship.

            It focuses on:
            - loading the train and test motion files
            - engineering telematics-style motion features
            - creating pseudo `trip_id` and `driver_id` fields for dashboarding
            - exporting cleaned outputs for later EDA, hypothesis testing, and violation analysis

            ## Important assumption

            The original dataset does **not** contain real `driver_id`, `trip_id`, ratings, or passenger feedback fields.
            To make the Week 4 dashboard possible, this notebook groups sequential sensor rows into fixed-size trips and assigns repeat pseudo drivers for analysis only.
            


In [ ]:
            !pip install pandas numpy matplotlib seaborn scipy statsmodels scikit-learn
            


In [ ]:
            from pathlib import Path
            import numpy as np
            import pandas as pd

            BASE_DIR = Path(r"/Users/ayet_dub/Documents/Codex/2026-05-11/files-mentioned-by-the-user-test")
            train_path = Path(r"/Users/ayet_dub/Library/CloudStorage/OneDrive-Personal/Lamina_Documentation/archive/train_motion_data.csv")
            test_path = Path(r"/Users/ayet_dub/Library/CloudStorage/OneDrive-Personal/Lamina_Documentation/archive/test_motion_data.csv")
            


In [ ]:
            def load_frame(path, split_name):
                df = pd.read_csv(path).copy()
                df["source_split"] = split_name
                return df

            train_df = load_frame(train_path, "train")
            test_df = load_frame(test_path, "test")
            raw_df = pd.concat([train_df, test_df], ignore_index=True)

            raw_df.head()
            


In [ ]:
            numeric_cols = ["AccX", "AccY", "AccZ", "GyroX", "GyroY", "GyroZ", "Timestamp"]
            df = raw_df.copy()

            for col in numeric_cols:
                df[col] = pd.to_numeric(df[col], errors="coerce")

            df = df.dropna(subset=numeric_cols + ["Class"]).copy()
            df["acc_mag"] = np.sqrt(df["AccX"]**2 + df["AccY"]**2 + df["AccZ"]**2)
            df["gyro_mag"] = np.sqrt(df["GyroX"]**2 + df["GyroY"]**2 + df["GyroZ"]**2)
            df["prev_acc_mag"] = df.groupby("source_split")["acc_mag"].shift(1)
            df["jerk_mag"] = (df["acc_mag"] - df["prev_acc_mag"]).abs().fillna(0)

            df.describe().T
            


In [ ]:
            trip_size = 60
            driver_count = 18
            class_offsets = {"AGGRESSIVE": 0, "NORMAL": 6, "SLOW": 12}

            df = df.sort_values(["source_split", "Timestamp"]).reset_index(drop=True)
            df["row_in_split"] = df.groupby("source_split").cumcount()
            df["trip_seq"] = df.groupby("source_split")["row_in_split"].transform(lambda s: s // trip_size)
            df["trip_id"] = df["source_split"].str.upper().str[:1] + "-TRIP-" + df["trip_seq"].add(1).astype(str).str.zfill(3)

            trip_lookup = (
                df.groupby("trip_id", as_index=False)
                .agg(
                    source_split=("source_split", "first"),
                    trip_seq=("trip_seq", "first"),
                    dominant_class=("Class", lambda s: s.mode().iat[0]),
                )
            )

            trip_lookup["driver_num"] = (
                trip_lookup["trip_seq"] + trip_lookup["dominant_class"].map(class_offsets).fillna(0).astype(int)
            ) % driver_count + 1
            trip_lookup["driver_id"] = trip_lookup["driver_num"].map(lambda x: f"D{x:03d}")

            df = df.merge(trip_lookup[["trip_id", "driver_id", "dominant_class"]], on="trip_id", how="left")
            df[["driver_id", "trip_id", "dominant_class"]].head()
            


In [ ]:
            output_path = BASE_DIR / "data" / "motion_sensor_enriched.csv"
            output_path.parent.mkdir(parents=True, exist_ok=True)
            df.to_csv(output_path, index=False)
            print("Saved:", output_path)
            
